# Classification Interactive Widgets

This notebook provides four interactive learning widgets for classification and logistic regression:
1. Decision Boundary Explorer
2. Sigmoid Function Explorer
3. Confusion Matrix Calculator
4. End-to-End Logistic Regression Pipeline

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

## Widget 1: Decision Boundary Explorer

In [2]:
def run_boundary_explorer(slope, intercept, class_a_count, class_b_count, seed):
    rng = np.random.default_rng(seed)

    # Features: x = participation, y = exam prep time
    a_x = rng.normal(70, 15, class_a_count)
    a_y = rng.normal(130, 15, class_a_count)
    b_x = rng.normal(35, 15, class_b_count)
    b_y = rng.normal(80, 15, class_b_count)

    x = np.concatenate([a_x, b_x])
    y = np.concatenate([a_y, b_y])
    labels = np.array([1] * class_a_count + [0] * class_b_count)

    y_boundary = slope * x + intercept
    preds = (y >= y_boundary).astype(int)

    correct = int((preds == labels).sum())
    total = len(labels)
    wrong = total - correct

    fig, ax = plt.subplots(figsize=(8.5, 6.5))

    xx = np.linspace(0, 100, 300)
    yy = slope * xx + intercept

    ax.fill_between(xx, yy, 170, color='#d9ecff', alpha=0.45, label='Predict A region')
    ax.fill_between(xx, -10, yy, color='#ffe8c4', alpha=0.45, label='Predict B region')

    ax.scatter(a_x, a_y, marker='*', s=120, color='#003D7C', label='Actual A')
    ax.scatter(b_x, b_y, marker='o', s=45, color='#FFA000', label='Actual B')
    ax.plot(xx, yy, '--', color='red', linewidth=2.0, label='Decision boundary')

    ax.set_xlim(0, 100)
    ax.set_ylim(0, 170)
    ax.set_xlabel('Assignment Participation Rate')
    ax.set_ylabel('Exam Preparation Time')
    ax.set_title('Decision Boundary Explorer')

    stats = f'Correct: {correct}  |  Misclassified: {wrong}  |  Accuracy: {correct/total:.2%}'
    ax.text(0.02, 0.02, stats, transform=ax.transAxes, fontsize=11,
            bbox=dict(boxstyle='round', fc='white', ec='gray', alpha=0.9))
    ax.legend(loc='upper left')
    plt.show()

boundary_slope = widgets.FloatSlider(description='Slope', min=-5.0, max=5.0, step=0.1, value=1.0)
boundary_intercept = widgets.FloatSlider(description='Intercept', min=-100, max=200, step=5, value=50)
boundary_a = widgets.IntSlider(description='Class A Count', min=5, max=50, step=1, value=20)
boundary_b = widgets.IntSlider(description='Class B Count', min=5, max=50, step=1, value=20)
boundary_seed = widgets.IntSlider(description='Seed', min=0, max=100, step=1, value=42)

boundary_out = widgets.interactive_output(
    run_boundary_explorer,
    {
        'slope': boundary_slope,
        'intercept': boundary_intercept,
        'class_a_count': boundary_a,
        'class_b_count': boundary_b,
        'seed': boundary_seed,
    }
)

display(widgets.VBox([boundary_slope, boundary_intercept, boundary_a, boundary_b, boundary_seed, boundary_out]))

## Widget 2: Sigmoid Function Explorer

In [3]:
def run_sigmoid_explorer(b0, b1, threshold, x_query):
    x = np.linspace(-10, 10, 400)
    z = b0 + b1 * x
    p = 1 / (1 + np.exp(-z))

    zq = b0 + b1 * x_query
    pq = 1 / (1 + np.exp(-zq))
    label = 'Class 1 (A)' if pq > threshold else 'Class 0 (B)'
    color = '#003D7C' if pq > threshold else '#FFA000'

    fig, ax = plt.subplots(figsize=(8.5, 6.0))
    ax.plot(x, p, color='red', linewidth=2.2, label='Sigmoid')
    ax.axhline(threshold, color='gray', linestyle='--', linewidth=2, label=f'Threshold={threshold:.2f}')

    ax.fill_between(x, threshold, 1, color='#d9ecff', alpha=0.3)
    ax.fill_between(x, 0, threshold, color='#ffe8c4', alpha=0.3)

    ax.scatter([x_query], [pq], s=120, color=color, edgecolor='black', zorder=4)

    txt = (
        f'z = b0 + b1*x1 = {zq:.3f}\n'
        f'P(Y=1) = {pq:.3f}\n'
        f'Classification: {label}'
    )
    ax.text(0.02, 0.04, txt, transform=ax.transAxes, fontsize=11,
            bbox=dict(boxstyle='round', fc='white', ec='gray', alpha=0.9))

    ax.set_xlim(-10, 10)
    ax.set_ylim(0, 1)
    ax.set_xlabel('x1')
    ax.set_ylabel('P(Y=1)')
    ax.set_title('Sigmoid Function Explorer')
    ax.legend(loc='lower right')
    plt.show()

sig_b0 = widgets.FloatSlider(description='b0', min=-10.0, max=10.0, step=0.1, value=0.0)
sig_b1 = widgets.FloatSlider(description='b1', min=-5.0, max=5.0, step=0.1, value=1.0)
sig_threshold = widgets.FloatSlider(description='Threshold', min=0.0, max=1.0, step=0.05, value=0.5)
sig_xq = widgets.FloatSlider(description='Query x1', min=-10.0, max=10.0, step=0.1, value=0.0)

sigmoid_out = widgets.interactive_output(
    run_sigmoid_explorer,
    {'b0': sig_b0, 'b1': sig_b1, 'threshold': sig_threshold, 'x_query': sig_xq}
)

display(widgets.VBox([sig_b0, sig_b1, sig_threshold, sig_xq, sigmoid_out]))

## Widget 3: Confusion Matrix Calculator

In [4]:
PRESETS = {
    'Custom': None,
    'Slide Example (Grades)': {'TP': 73, 'FP': 1, 'FN': 3, 'TN': 23},
    'Slide Example (Spam)': {'TP': 70, 'FP': 30, 'FN': 80, 'TN': 70},
}

def safe_div(a, b):
    return None if b == 0 else a / b

cm_tp = widgets.IntSlider(description='TP', min=0, max=200, value=73)
cm_fp = widgets.IntSlider(description='FP', min=0, max=200, value=1)
cm_fn = widgets.IntSlider(description='FN', min=0, max=200, value=3)
cm_tn = widgets.IntSlider(description='TN', min=0, max=200, value=23)
cm_preset = widgets.Dropdown(description='Preset', options=list(PRESETS.keys()), value='Custom')
cm_out = widgets.Output()

def render_cm(*_):
    with cm_out:
        clear_output(wait=True)

        tp = cm_tp.value
        fp = cm_fp.value
        fn = cm_fn.value
        tn = cm_tn.value

        matrix = np.array([[tn, fp], [fn, tp]])

        total = tp + fp + fn + tn
        acc = safe_div(tp + tn, total)
        prec = safe_div(tp, tp + fp)
        rec = safe_div(tp, tp + fn)
        f1 = None if (prec is None or rec is None or (prec + rec) == 0) else 2 * prec * rec / (prec + rec)

        fig, ax = plt.subplots(figsize=(5.0, 4.5))
        ax.imshow([[0, 1], [1, 0]], cmap='RdYlGn', alpha=0.25)

        labels = [['TN', 'FP'], ['FN', 'TP']]
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{labels[i][j]}\n{matrix[i,j]}", ha='center', va='center', fontsize=13, weight='bold')

        ax.set_xticks([0, 1], labels=['Pred 0', 'Pred 1'])
        ax.set_yticks([0, 1], labels=['Actual 0', 'Actual 1'])
        ax.set_title('Confusion Matrix')
        ax.grid(False)
        plt.show()

        fmt = lambda v: 'N/A' if v is None else f'{v:.4f}'
        pct = lambda v: 'N/A' if v is None else f'{v:.2%}'

        print(f'Accuracy  = (TP + TN) / Total = {fmt(acc)}  ({pct(acc)})')
        print(f'Precision = TP / (TP + FP)    = {fmt(prec)}')
        print(f'Recall    = TP / (TP + FN)    = {fmt(rec)}')
        print(f'F1-Score  = 2*P*R / (P + R)   = {fmt(f1)}')

def on_preset_change(change):
    if change['name'] == 'value' and change['new'] in PRESETS and PRESETS[change['new']] is not None:
        vals = PRESETS[change['new']]
        cm_tp.value = vals['TP']
        cm_fp.value = vals['FP']
        cm_fn.value = vals['FN']
        cm_tn.value = vals['TN']

def on_slider_change(change):
    if change['name'] == 'value' and cm_preset.value != 'Custom':
        cm_preset.value = 'Custom'

cm_preset.observe(on_preset_change, names='value')
for s in [cm_tp, cm_fp, cm_fn, cm_tn]:
    s.observe(on_slider_change, names='value')
    s.observe(render_cm, names='value')
cm_preset.observe(render_cm, names='value')

display(widgets.VBox([cm_preset, cm_tp, cm_fp, cm_fn, cm_tn, cm_out]))
render_cm()

## Widget 4: End-to-End Logistic Regression Pipeline

In [5]:
pipe_n = widgets.IntSlider(description='Samples', min=50, max=500, step=10, value=200)
pipe_test = widgets.FloatSlider(description='Test size', min=0.10, max=0.50, step=0.05, value=0.25)
pipe_seed = widgets.IntSlider(description='Random state', min=0, max=100, step=1, value=42)
pipe_sep = widgets.FloatSlider(description='Class separation', min=0.5, max=5.0, step=0.1, value=2.0)
pipe_btn = widgets.Button(description='Run Pipeline', button_style='primary')
pipe_status = widgets.Label(value='Ready')
pipe_out = widgets.Output()

def run_pipeline(_):
    with pipe_out:
        clear_output(wait=True)
        pipe_status.value = 'Training...'

        X, y = make_classification(
            n_samples=pipe_n.value,
            n_features=2,
            n_redundant=0,
            n_informative=2,
            n_clusters_per_class=1,
            class_sep=pipe_sep.value,
            random_state=pipe_seed.value,
        )

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=pipe_test.value, random_state=pipe_seed.value
        )

        model = LogisticRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        cm = confusion_matrix(y_test, y_pred)

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
        y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
        zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        axes[0, 0].contourf(xx, yy, zz, alpha=0.22, cmap='coolwarm')
        axes[0, 0].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=22)
        axes[0, 0].set_title('Train Set + Decision Boundary')
        axes[0, 0].set_xlabel('Exam Prep Time')
        axes[0, 0].set_ylabel('Participation Rate')

        axes[0, 1].contourf(xx, yy, zz, alpha=0.22, cmap='coolwarm')
        axes[0, 1].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm', s=30)
        wrong = y_test != y_pred
        axes[0, 1].scatter(X_test[wrong, 0], X_test[wrong, 1], facecolors='none', edgecolors='red', s=120, linewidths=1.8)
        axes[0, 1].set_title('Test Set (misclassified circled)')
        axes[0, 1].set_xlabel('Exam Prep Time')
        axes[0, 1].set_ylabel('Participation Rate')

        axes[1, 0].imshow(cm, cmap='Blues')
        for i in range(2):
            for j in range(2):
                axes[1, 0].text(j, i, cm[i, j], ha='center', va='center', fontsize=12, color='black')
        axes[1, 0].set_xticks([0, 1], labels=['Pred 0', 'Pred 1'])
        axes[1, 0].set_yticks([0, 1], labels=['Actual 0', 'Actual 1'])
        axes[1, 0].set_title('Confusion Matrix')

        axes[1, 1].axis('off')
        metrics_txt = (
            f'Accuracy : {acc:.4f}\n'
            f'Precision: {prec:.4f}\n'
            f'Recall   : {rec:.4f}\n'
            f'F1-Score : {f1:.4f}\n\n'
            f'b0 (intercept): {model.intercept_[0]:.4f}\n'
            f'b1: {model.coef_[0,0]:.4f}\n'
            f'b2: {model.coef_[0,1]:.4f}'
        )
        axes[1, 1].text(0.02, 0.98, metrics_txt, va='top', fontsize=12)
        axes[1, 1].set_title('Model Metrics')

        plt.tight_layout()
        plt.show()

        print('Classification Report:')
        print(classification_report(y_test, y_pred, zero_division=0))
        pipe_status.value = 'Done'

pipe_btn.on_click(run_pipeline)
display(widgets.VBox([pipe_n, pipe_test, pipe_seed, pipe_sep, widgets.HBox([pipe_btn, pipe_status]), pipe_out]))